# 04 — Shoreline Quality Assurance

This notebook checks the corrected 2024 Mombasa shoreline candidates and supports visual review in QGIS. It does not create transects or calculate shoreline change. Automated checks alone do not confirm shoreline accuracy.


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

cwd = Path.cwd()
if (cwd / "config").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "config").exists() and (cwd.parent / "notebooks").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not resolve PROJECT_ROOT")

STUDY_AREA_PATH = PROJECT_ROOT / "config" / "study_area.geojson"
SCENE_METADATA_PATH = (
    PROJECT_ROOT / "data" / "interim" / "mombasa_test" / "selected_scene_metadata.csv"
)
WATER_MASK_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "mombasa_test"
    / "water_mask"
    / "water_mask_2024.tif"
)
MNDWI_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "mombasa_test"
    / "water_index"
    / "mndwi_2024.tif"
)
SHORELINE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mombasa_test"
    / "2024"
    / "shoreline_2024_shared_boundary_test.gpkg"
)
QA_DIR = PROJECT_ROOT / "outputs" / "qa" / "mombasa_test" / "2024"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
SHORELINE_LAYER = "shoreline_candidates"

for d in (QA_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
for name, path in [
    ("STUDY_AREA_PATH", STUDY_AREA_PATH),
    ("SCENE_METADATA_PATH", SCENE_METADATA_PATH),
    ("WATER_MASK_PATH", WATER_MASK_PATH),
    ("MNDWI_PATH", MNDWI_PATH),
    ("SHORELINE_PATH", SHORELINE_PATH),
]:
    print(f"{name}: {path} (exists={path.exists()})")

PROJECT_ROOT: [redacted env]\Documents\Data Science Projects\Shoremonitor
STUDY_AREA_PATH: [redacted env]\Documents\Data Science Projects\Shoremonitor\config\study_area.geojson (exists=True)
SCENE_METADATA_PATH: [redacted env]\Documents\Data Science Projects\Shoremonitor\data\interim\mombasa_test\selected_scene_metada
ta.csv (exists=True)
WATER_MASK_PATH: [redacted env]\Documents\Data Science Projects\Shoremonitor\data\interim\mombasa_test\water_mask\water_mask_202
4.tif (exists=True)
MNDWI_PATH: [redacted env]\Documents\Data Science Projects\Shoremonitor\data\interim\mombasa_test\water_index\mndwi_2024.tif (ex
ists=True)
SHORELINE_PATH: [redacted env]\Documents\Data Science Projects\Shoremonitor\data\processed\mombasa_test\2024\shoreline_2024_shar
ed_boundary_test.gpkg (exists=True)


## Verify required inputs and load corrected shoreline

Check that all required files exist, confirm the shoreline layer, and load the corrected candidates for QA without modifying them.


In [7]:
missing = [
    str(p)
    for p in (
        STUDY_AREA_PATH,
        SCENE_METADATA_PATH,
        WATER_MASK_PATH,
        MNDWI_PATH,
        SHORELINE_PATH,
    )
    if not p.exists()
]
if missing:
    raise FileNotFoundError("Missing required input(s):\n" + "\n".join(missing))

try:
    layers_table = gpd.list_layers(SHORELINE_PATH)
    print(layers_table)
    available_layers = layers_table["name"].tolist()
except Exception:
    import fiona

    available_layers = fiona.listlayers(str(SHORELINE_PATH))
print("Available layers:", available_layers)
if SHORELINE_LAYER not in available_layers:
    raise ValueError(
        f"Layer '{SHORELINE_LAYER}' not found. Available: {available_layers}"
    )

shoreline_candidates = gpd.read_file(SHORELINE_PATH, layer=SHORELINE_LAYER)
study_area = gpd.read_file(STUDY_AREA_PATH)
scene_metadata = pd.read_csv(SCENE_METADATA_PATH)

if shoreline_candidates.crs is None:
    raise ValueError("Shoreline CRS is missing")
if study_area.crs is None:
    raise ValueError("Study area CRS is missing")

for label, path in [("WATER_MASK", WATER_MASK_PATH), ("MNDWI", MNDWI_PATH)]:
    with rasterio.open(path) as src:
        print(label, "CRS:", src.crs)
        print(label, "Width/Height:", src.width, src.height)
        print(label, "Resolution:", src.res)
        print(label, "Bounds:", src.bounds)
        print(label, "NoData:", src.nodata)

print("Shoreline feature count:", len(shoreline_candidates))
print("Shoreline CRS:", shoreline_candidates.crs)
print(
    "Geometry types:", shoreline_candidates.geometry.geom_type.value_counts().to_dict()
)
print(shoreline_candidates.drop(columns=["geometry"]).head())
print(scene_metadata)

reported_count = 15802
if len(shoreline_candidates) != reported_count:
    print(
        f"WARNING: loaded count {len(shoreline_candidates)} differs from reported {reported_count}"
    )

                   name geometry_type
0  shoreline_candidates    LineString
Available layers: ['shoreline_candidates']
WATER_MASK CRS: EPSG:32737
WATER_MASK Width/Height: 2222 2213
WATER_MASK Resolution: (10.0, 10.0)
WATER_MASK Bounds: BoundingBox(left=572150.0, bottom=9546770.0, right=594370.0, top=9568900.0)
WATER_MASK NoData: None
MNDWI CRS: EPSG:32737
MNDWI Width/Height: 2222 2213
MNDWI Resolution: (10.0, 10.0)
MNDWI Bounds: BoundingBox(left=572150.0, bottom=9546770.0, right=594370.0, top=9568900.0)
MNDWI NoData: None
Shoreline feature count: 15802
Shoreline CRS: EPSG:32737
Geometry types: {'LineString': 15802}
   shoreline_id                                       source_scene  \
0             1  S2A_MSIL2A_20240529T074541_R049_T37MER_2024052...
1             2  S2A_MSIL2A_20240529T074541_R049_T37MER_2024052...
2             3  S2A_MSIL2A_20240529T074541_R049_T37MER_2024052...
3             4  S2A_MSIL2A_20240529T074541_R049_T37MER_2024052...
4             5  S2A_MSIL2A_20240529T07

## Verify raster alignment and invalid pixels

Confirm that water mask and MNDWI grids align and that invalid pixels remain distinguishable.


In [8]:
with rasterio.open(WATER_MASK_PATH) as wm_src, rasterio.open(MNDWI_PATH) as mndwi_src:
    water_values = wm_src.read(1, masked=True)
    mndwi_values = mndwi_src.read(1, masked=True)
    water_mask_arr = np.ma.getmaskarray(water_values)
    mndwi_mask_arr = np.ma.getmaskarray(mndwi_values)
    wm_crs, mndwi_crs = wm_src.crs, mndwi_src.crs
    wm_shape, mndwi_shape = (
        (wm_src.height, wm_src.width),
        (mndwi_src.height, mndwi_src.width),
    )
    wm_transform, mndwi_transform = wm_src.transform, mndwi_src.transform
    wm_bounds, mndwi_bounds = wm_src.bounds, mndwi_src.bounds

print("CRS match:", wm_crs == mndwi_crs, wm_crs, mndwi_crs)
print("Shape match:", wm_shape == mndwi_shape, wm_shape, mndwi_shape)
print("Transform match:", wm_transform == mndwi_transform)
print("Bounds match:", wm_bounds == mndwi_bounds)

if not (
    wm_crs == mndwi_crs
    and wm_shape == mndwi_shape
    and wm_transform == mndwi_transform
    and wm_bounds == mndwi_bounds
):
    print("Grid mismatch detected; stopping pixel comparison")
    raise SystemExit

water_finite = np.isfinite(water_values.filled(np.nan))
water_valid = (~water_mask_arr) & water_finite
unique_water = np.unique(water_values.filled(np.nan)[water_valid])
print("Unique valid water-mask values:", unique_water)
if not set(unique_water.tolist()).issubset({0, 1}):
    raise ValueError("Unexpected valid water-mask values")

water_pixels_mask = water_valid & (water_values.filled(np.nan) == 1)
land_pixels_mask = water_valid & (water_values.filled(np.nan) == 0)
invalid_pixels_mask = ~water_valid

total_pixels = water_values.size
water_count = int(water_pixels_mask.sum())
land_count = int(land_pixels_mask.sum())
invalid_count = int(invalid_pixels_mask.sum())
valid_total = water_count + land_count
print("Total:", total_pixels)
print("Water:", water_count)
print("Land:", land_count)
print("Invalid:", invalid_count)
print("Valid %:", valid_total / total_pixels * 100)
print("Water % of valid:", (water_count / valid_total * 100) if valid_total else 0)
print("Land % of valid:", (land_count / valid_total * 100) if valid_total else 0)
assert water_count + land_count + invalid_count == total_pixels

for name, actual, expected in [
    ("Water", water_count, 2952072),
    ("Land", land_count, 1870198),
    ("Invalid", invalid_count, 95016),
]:
    print(
        name,
        "MATCH"
        if actual == expected
        else f"MISMATCH actual={actual} expected={expected}",
    )

water_valid_mndwi_invalid = int(
    (
        water_valid
        & (~np.isfinite(mndwi_values.filled(np.nan)) | np.ma.getmaskarray(mndwi_values))
    ).sum()
)
invalid_water_valid_mndwi = int(
    (
        invalid_pixels_mask
        & np.isfinite(mndwi_values.filled(np.nan))
        & (~np.ma.getmaskarray(mndwi_values))
    ).sum()
)
print("Valid water-mask pixels with invalid MNDWI:", water_valid_mndwi_invalid)
print("Invalid water-mask pixels with valid MNDWI:", invalid_water_valid_mndwi)
print("Invalid pixels in water mask:", int(invalid_pixels_mask.sum()))
print(
    "Invalid pixels in MNDWI:",
    int(
        (
            ~np.isfinite(mndwi_values.filled(np.nan)) | np.ma.getmaskarray(mndwi_values)
        ).sum()
    ),
)

if invalid_count == 0:
    print("WARNING: no invalid pixels found, conflicts with previous 95,016 count")
else:
    print("Invalid representation: stored mask and/or nonfinite values")

CRS match: True EPSG:32737 EPSG:32737
Shape match: True (2213, 2222) (2213, 2222)
Transform match: True
Bounds match: True
Unique valid water-mask values: [0. 1.]
Total: 4917286
Water: 2952072
Land: 1870198
Invalid: 95016
Valid %: 98.06771458890128
Water % of valid: 61.217476416708315
Land % of valid: 38.782523583291685
Water MATCH
Land MATCH
Invalid MATCH
Valid water-mask pixels with invalid MNDWI: 0
Invalid water-mask pixels with valid MNDWI: 0
Invalid pixels in water mask: 95016
Invalid pixels in MNDWI: 95016
Invalid representation: stored mask and/or nonfinite values


## Inspect geometry quality and segment lengths

Assess whether the large feature count is driven by short line segments. No geometries are modified.


In [9]:
if shoreline_candidates.crs is None:
    raise ValueError("CRS is missing")
units = (
    str(shoreline_candidates.crs.axis_info[0].unit_name)
    if hasattr(shoreline_candidates.crs, "axis_info")
    else ""
)
epsg = shoreline_candidates.crs.to_epsg()
print("EPSG:", epsg, "Units:", units)
if "metre" not in units.lower() and "meter" not in units.lower():
    raise ValueError("CRS horizontal units are not metres")

print("Total features:", len(shoreline_candidates))
print("Null geometries:", shoreline_candidates.geometry.isna().sum())
print("Empty geometries:", shoreline_candidates.geometry.is_empty.sum())
print(
    "Invalid nonempty:",
    (
        ~shoreline_candidates.geometry.is_valid
        & ~shoreline_candidates.geometry.is_empty
        & shoreline_candidates.geometry.notna()
    ).sum(),
)
print(
    "Geometry types:", shoreline_candidates.geometry.geom_type.value_counts().to_dict()
)

inspect = shoreline_candidates[
    shoreline_candidates.geometry.notna()
    & ~shoreline_candidates.geometry.is_empty
    & shoreline_candidates.geometry.is_valid
].copy()
excluded = len(shoreline_candidates) - len(inspect)
print("Excluded features:", excluded)
if (
    ~shoreline_candidates.geometry.is_valid
    & shoreline_candidates.geometry.notna()
    & ~shoreline_candidates.geometry.is_empty
).any():
    print("QA issue: invalid geometries present")

inspect["orig_id"] = inspect.index
exploded = inspect.explode(index_parts=False).reset_index(drop=True)
exploded["length_m"] = exploded.geometry.length

print("Line parts:", len(exploded))
print("Zero-length parts:", int((exploded["length_m"] == 0).sum()))
print("Total km:", round(float(exploded["length_m"].sum()) / 1000, 4))
print("Min:", float(exploded["length_m"].min()) if len(exploded) else 0)
print("25th:", float(exploded["length_m"].quantile(0.25)) if len(exploded) else 0)
print("Median:", float(exploded["length_m"].median()) if len(exploded) else 0)
print("75th:", float(exploded["length_m"].quantile(0.75)) if len(exploded) else 0)
print("Max:", float(exploded["length_m"].max()) if len(exploded) else 0)

bins = [0, 10, 20, 50, 100, 500, np.inf]
labels = ["0-10", "10-20", "20-50", "50-100", "100-500", ">500"]
exploded["bin"] = pd.cut(
    exploded["length_m"], bins=bins, labels=labels, include_lowest=True
)
table = (
    exploded.groupby("bin", observed=True)
    .agg(count=("length_m", "size"), total_km=("length_m", lambda s: s.sum() / 1000))
    .reset_index()
)
table["pct"] = table["count"] / len(exploded) * 100 if len(exploded) else 0
print(table.to_string(index=False))

dup_count = exploded.geometry.duplicated().sum()
print("Exact duplicate geometries (coordinate duplicate check):", int(dup_count))

measured_km = float(exploded["length_m"].sum()) / 1000
if abs(measured_km - 265.95) > 0.01:
    print(f"Discrepancy: measured {measured_km:.4f} km vs 265.95 km")
else:
    print("Length within 0.01 km tolerance")

print("Ten shortest:")
print(
    exploded.nsmallest(10, "length_m")[["orig_id", "length_m"]].to_string(index=False)
)
print("Ten longest:")
print(exploded.nlargest(10, "length_m")[["orig_id", "length_m"]].to_string(index=False))

EPSG: 32737 Units: metre
Total features: 15802
Null geometries: 0
Empty geometries: 0
Invalid nonempty: 0
Geometry types: {'LineString': 15802}
Excluded features: 0
Line parts: 15802
Zero-length parts: 0
Total km: 265.95
Min: 10.0
25th: 10.0
Median: 10.0
75th: 20.0
Max: 340.0
    bin  count  total_km       pct
   0-10  10125    101.25 64.074168
  10-20   3785     75.70 23.952664
  20-50   1476     53.19  9.340590
 50-100    346     24.83  2.189596
100-500     70     10.98  0.442982
Exact duplicate geometries (coordinate duplicate check): 0
Length within 0.01 km tolerance
Ten shortest:
 orig_id  length_m
       1      10.0
       2      10.0
       3      10.0
       5      10.0
       7      10.0
       9      10.0
      10      10.0
      11      10.0
      13      10.0
      15      10.0
Ten longest:
 orig_id  length_m
      94     340.0
    9966     320.0
   15180     270.0
    1345     260.0
    3889     260.0
    1293     250.0
    4801     230.0
    7923     230.0
      62     20